In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("cs-training.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Shape: (150000, 12)

Columns:
['Unnamed: 0', 'SeriousDlqin2yrs', 'RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents']


,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


## Missing-value analysis

In [2]:
missing = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": df.isnull().mean() * 100
})

missing = missing[missing["Missing Count"] > 0]
missing.sort_values("Missing Percentage", ascending=False)

,Missing Count,Missing Percentage
MonthlyIncome,29731,19.820667
NumberOfDependents,3924,2.616000


we should inspect distributions and relationships, not blindly calculate mean, median and mode for every category and fill accordingly.
Here's how we'll decide
1. MonthlyIncome
We'll examine:
- Mean, median and mode.
- Distribution and skewness.
- Outliers.
- Income across categories such as age groups, number of dependents and number of open credit lines.
- Whether missing income is associated with SeriousDlqin2yrs.
If income is highly skewed, median imputation may be more appropriate than mean imputation.

We could also investigate conditional median imputation, such as using age groups or other relevant characteristics, but only if the groups have enough observations and the approach is justified.

2. NumberOfDependents
We'll examine:
- Mean, median and mode.
- Frequency distribution (0, 1, 2, 3, etc.).
- Distribution across age groups and income groups.
- Whether missingness is associated with delinquency.
Since this is a discrete count, mode or median may be reasonable candidates.

###  inspect

In [3]:
for col in ["MonthlyIncome", "NumberOfDependents"]:
    print(f"\n{'='*45}")
    print(col)
    print(df[col].describe())
    print("\nMode:")
    print(df[col].mode().tolist())
    print("\nSkewness:", df[col].skew())


MonthlyIncome
count    1.202690e+05
mean     6.670221e+03
std      1.438467e+04
min      0.000000e+00
25%      3.400000e+03
50%      5.400000e+03
75%      8.249000e+03
max      3.008750e+06
Name: MonthlyIncome, dtype: float64

Mode:
[5000.0]

Skewness: 114.0403179452332

NumberOfDependents
count    146076.000000
mean          0.757222
std           1.115086
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max          20.000000
Name: NumberOfDependents, dtype: float64

Mode:
[0.0]

Skewness: 1.5882423788858833


- Mean: 6,670
- Median: 5,400
- Mode: 5,000
- Skewness: 114.04
- Maximum: 3,008,750
The mean is considerably higher than the median because a small number of extremely large income values pull it upwards.

Initial conclusion: Median is preferable to mean as a basic imputation candidate.

But we should investigate whether income differs substantially across meaningful groups before deciding whether conditional imputation is worthwhile.

NumberOfDependents
- Mean: 0.76
- Median: 0
- Mode: 0
- Skewness: 1.59
- Maximum: 20
At least 50% of the observed records have zero dependents.

Initial conclusion: Median and mode are both 0. However, we should inspect the distribution before choosing between them.

## Next: inspect the distributions
- see the actual frequency patterns.

In [4]:
for col in ["MonthlyIncome", "NumberOfDependents"]:
    print(f"\n{col}")
    print(df[col].value_counts().head(15))
    print("\nQuantiles:")
    print(df[col].quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))


MonthlyIncome
5000.0     2757
4000.0     2106
6000.0     1934
3000.0     1758
0.0        1634
2500.0     1551
10000.0    1466
3500.0     1360
4500.0     1226
7000.0     1223
8000.0     1127
2000.0     1116
7500.0     1051
10500.0    1013
5500.0      897
Name: MonthlyIncome, dtype: int64

Quantiles:
0.10     2005.0
0.25     3400.0
0.50     5400.0
0.75     8249.0
0.90    11666.0
0.95    14587.6
0.99    25000.0
Name: MonthlyIncome, dtype: float64

NumberOfDependents
0.0     86902
1.0     26316
2.0     19522
3.0      9483
4.0      2862
5.0       746
6.0       158
7.0        51
8.0        24
10.0        5
9.0         5
20.0        1
13.0        1
Name: NumberOfDependents, dtype: int64

Quantiles:
0.10    0.0
0.25    0.0
0.50    0.0
0.75    1.0
0.90    2.0
0.95    3.0
0.99    4.0
Name: NumberOfDependents, dtype: float64


## examine group-wise patterns

In [5]:
df["AgeGroup"] = pd.cut(
    df["age"],
    bins=[0, 25, 35, 45, 55, 65, 100],
    labels=["18-25", "26-35", "36-45", "46-55", "56-65", "65+"]
)

group_analysis = df.groupby("AgeGroup", observed=False).agg(
    Income_Median=("MonthlyIncome", "median"),
    Income_Mean=("MonthlyIncome", "mean"),
    Income_Missing=("MonthlyIncome", lambda x: x.isna().mean() * 100),
    Dependents_Median=("NumberOfDependents", "median"),
    Dependents_Missing=("NumberOfDependents", lambda x: x.isna().mean() * 100),
    Records=("age", "count")
)

group_analysis.round(2)

,Income_Median,Income_Mean,Income_Missing,Dependents_Median,Dependents_Missing,Records
AgeGroup,,,,,,
18-25,1600.0,1992.73,19.95,0.0,6.97,3027
26-35,3876.0,4554.99,13.48,0.0,1.90,18458
36-45,5611.0,6759.94,14.41,1.0,0.96,29819
46-55,6267.0,7644.72,17.38,1.0,1.74,36690
56-65,6175.5,7571.53,22.87,0.0,2.56,33406
65+,4972.5,6181.26,29.10,0.0,5.54,28586


- Income medians vary substantially across age groups.
- Missing income is particularly frequent in the 18–25 and 65+ groups.
- The mean is consistently higher than the median, reinforcing the right-skewness we observed.

## age is also a sensitive attribute for our fairness evaluation. Using it to impute income could introduce or reinforce age-related patterns. We'll measure that effect rather than assume it's harmless.

## inspect the extreme records

In [6]:
df.nlargest(15, "MonthlyIncome")[
    ["MonthlyIncome", "age", "DebtRatio",
     "NumberOfDependents", "SeriousDlqin2yrs"]
]

,MonthlyIncome,age,DebtRatio,NumberOfDependents,SeriousDlqin2yrs
73763,3008750.0,52,0.001470,3.0,0
137140,1794060.0,68,0.002776,0.0,0
111365,1560100.0,44,0.004013,2.0,0
50640,1072500.0,44,0.004537,1.0,0
122543,835040.0,55,0.000147,0.0,0
123291,730483.0,67,0.005570,0.0,0
93564,702500.0,50,0.008268,2.0,0
96549,699530.0,52,0.003698,1.0,0
119136,649587.0,49,0.001313,3.0,0
37078,629000.0,83,0.000000,0.0,0


Feature	Value
MonthlyIncome	3,008,750
Age	52
DebtRatio	0.001470
Dependents	3
Delinquency target	0

The very high income is accompanied by a very low debt ratio.

This is plausible, but we cannot establish that the income is correct from these columns alone.

Also, notice that the high-income records have different ages and numbers of dependents. We shouldn't automatically exclude an entire age or income group.

## investigate whether missingness relates to delinquency

In [7]:
for col in ["MonthlyIncome", "NumberOfDependents"]:
    print(f"\n{col}")

    print(
        df.groupby(df[col].isna())["SeriousDlqin2yrs"]
        .agg(["count", "mean"])
        .rename(index={False: "Available", True: "Missing"})
    )


MonthlyIncome
                count      mean
MonthlyIncome                  
Available      120269  0.069486
Missing         29731  0.056137

NumberOfDependents
                     count      mean
NumberOfDependents                  
Available           146076  0.067410
Missing               3924  0.045617


### missing values are not distributed equally with respect to the target.

## investigate the upper tail

In [8]:
income = df["MonthlyIncome"]

for percentile in [0.95, 0.99, 0.995, 0.999]:
    print(
        f"{percentile * 100}% percentile:",
        income.quantile(percentile)
    )

print("\nRecords above 25,000:", (income > 25000).sum())
print("Records above 50,000:", (income > 50000).sum())
print("Records above 100,000:", (income > 100000).sum())
print("Records above 500,000:", (income > 500000).sum())

95.0% percentile: 14587.599999999991
99.0% percentile: 25000.0
99.5% percentile: 35000.0
99.9% percentile: 78395.74800000706

Records above 25,000: 1168
Records above 50,000: 301
Records above 100,000: 70
Records above 500,000: 12


- Only 1,168 records have income above 25,000.
- Just 70 exceed 100,000.
- Twelve exceed 500,000.

### calculate the difference

In [9]:
for col in ["MonthlyIncome", "NumberOfDependents"]:
    rates = df.groupby(df[col].isna())["SeriousDlqin2yrs"].mean()

    print(f"\n{col}")
    print("Available:", rates[False])
    print("Missing:", rates[True])
    print("Difference:", rates[False] - rates[True])


MonthlyIncome
Available: 0.06948590243537403
Missing: 0.05613669234132723
Difference: 0.013349210094046798

NumberOfDependents
Available: 0.0674101152824557
Missing: 0.04561671763506626
Difference: 0.021793397647389447


- use this to decide whether missing-value indicators deserve to be tested alongside imputation.
- highly skewed income distribution and target-associated missingness.

## inspect income missingness across income-independent features

In [10]:
df.groupby("AgeGroup", observed=False).agg(
    Records=("age", "count"),
    Income_Missing=("MonthlyIncome", lambda x: x.isna().sum()),
    Income_Missing_Percent=("MonthlyIncome", lambda x: x.isna().mean() * 100),
    Delinquency_Rate=("SeriousDlqin2yrs", "mean")
).round(3)

,Records,Income_Missing,Income_Missing_Percent,Delinquency_Rate
AgeGroup,,,,
18-25,3027,604,19.954,0.112
26-35,18458,2488,13.479,0.111
36-45,29819,4296,14.407,0.088
46-55,36690,6378,17.383,0.076
56-65,33406,7640,22.870,0.046
65+,28586,8318,29.098,0.024


## Income missingness: The 65+ group has the highest missing-income rate (29.10%), compared with 13.48% for ages 26–35.
- Delinquency: The rate falls from 11.2% among ages 18–25 to 2.4% among those aged 65+.
- Potential relationship: Age is associated with both income missingness and delinquency in this dataset.

## inspect income distribution across age groups

In [11]:
df.groupby("AgeGroup", observed=False)["MonthlyIncome"].agg(
    ["count", "median", "mean", "min", "max"]
).round(2)

,count,median,mean,min,max
AgeGroup,,,,,
18-25,2423,1600.0,1992.73,0.0,36000.0
26-35,15970,3876.0,4554.99,0.0,251608.0
36-45,25523,5611.0,6759.94,0.0,1560100.0
46-55,30312,6267.0,7644.72,0.0,3008750.0
56-65,25766,6175.5,7571.53,0.0,582369.0
65+,20268,4972.5,6181.26,0.0,1794060.0
